# WARO API - Python Usage Examples

This notebook contains step-by-step examples to connect and query the WARO Colombia API.

**Base URL:** `https://api.warocol.com/v1`

---

## 1. Install Dependencies

First, install the `requests` library if you don't have it:

In [ ]:
# Install requests (run only if not installed)
!pip install requests

## 2. Initial Configuration

Set up your API Token and base functions to make requests.

**Get your API Token:** Go to [https://warocol.com/integraciones](https://warocol.com/integraciones) to create your API Key.

In [ ]:
import requests
import json
from datetime import datetime, timedelta

# =============================================================================
# CONFIGURATION - Replace with your API Token
# Get your token at: https://warocol.com/integraciones
# =============================================================================
API_TOKEN = "waro_sk_YOUR_TOKEN_HERE"  # <-- Put your token here
BASE_URL = "https://api.warocol.com/v1"

# Authentication headers
HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

def make_request(endpoint, data=None):
    """
    Helper function to make POST requests to the API.
    
    Args:
        endpoint: The endpoint to query (e.g., '/sales')
        data: Dictionary with request parameters
    
    Returns:
        dict: API response in JSON format
    """
    url = f"{BASE_URL}{endpoint}"
    response = requests.post(url, headers=HEADERS, json=data or {})
    return response.json()

print("Configuration ready!")

---
## 3. Get Sales List

**Endpoint:** `POST /v1/sales`

Retrieves a paginated list of sales/orders.

**Required Scope:** `orders:read` or `read`

### Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `limit` | int | 50 | Number of results to return (1-250) |
| `offset` | int | 0 | Number of results to skip for pagination |
| `paymentMethod` | string | null | Filter by payment method: `"cash"`, `"card"`, `"digital"` |
| `status` | string | null | Filter by order status: `"completed"`, `"cancelled"`, `"pending"` |
| `sortField` | string | "order_date" | Field to sort by |
| `sortDirection` | string | "desc" | Sort direction: `"asc"` or `"desc"` |
| `dateFrom` | string | null | Start date filter (format: YYYY-MM-DD) |
| `dateTo` | string | null | End date filter (format: YYYY-MM-DD) |
| `timezone` | string | "America/Bogota" | IANA timezone for date filters |

In [ ]:
# Example 1: Get the last 10 sales
result = make_request("/sales", {
    "limit": 10,
    "offset": 0,
    "sortField": "order_date",
    "sortDirection": "desc"
})

if result.get("success"):
    print(f"Total sales: {result['pagination']['total']}")
    print(f"Showing: {len(result['data'])} sales\n")
    
    for sale in result['data']:
        print(f"- Order #{sale.get('orderNumber', 'N/A')} | Date: {sale.get('orderDate', 'N/A')}")
else:
    print(f"Error: {result.get('error', {}).get('message', 'Unknown error')}")

In [ ]:
# Example 2: Filter sales by date and payment method
# Completed cash sales from the last 7 days

date_from = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
date_to = datetime.now().strftime("%Y-%m-%d")

result = make_request("/sales", {
    "limit": 50,
    "dateFrom": date_from,
    "dateTo": date_to,
    "paymentMethod": "cash",
    "status": "completed",
    "timezone": "America/Bogota"
})

if result.get("success"):
    print(f"Cash sales from {date_from} to {date_to}:")
    print(f"Total found: {result['pagination']['total']}")
else:
    print(f"Error: {result.get('error', {}).get('message')}")

---
## 4. Get Sales Metrics

**Endpoint:** `POST /v1/sales/metrics`

Retrieves aggregated sales metrics for the authenticated tenant.

**Required Scope:** `orders:read` or `read`

### Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `dateFrom` | string | null | Start date filter (format: YYYY-MM-DD) |
| `dateTo` | string | null | End date filter (format: YYYY-MM-DD) |
| `timezone` | string | "America/Bogota" | IANA timezone for date filters |
| `groupBy` | string | null | Grouping option — see table below |
| `limit` | int | 20 | For `groupBy: "product"` only — number of products (1-100) |
| `sortBy` | string | "quantity" | For `groupBy: "product"` only — `"quantity"` or `"revenue"` |
| `ranges` | array | null | For `groupBy: "ticket"` only — custom price breakpoints (e.g. `[0, 15000, 25000, 40000]`) |

### groupBy Options

| `groupBy` | Description | Extra params |
|-----------|-------------|--------------|
| `null` (default) | Overall totals for the period | — |
| `"date"` | Day-by-day breakdown | — |
| `"weekday"` | Aggregated by day of week (Mon–Sun) | — |
| `"hour"` | Aggregated by hour of day (0–23) | — |
| `"product"` | Top selling products | `limit`, `sortBy` |
| `"payment"` | Breakdown by payment method | — |
| `"ticket"` | Distribution by ticket price ranges | `ranges` |

### Response Fields

**`groupBy: null` (default)**

| Field | Type | Description |
|-------|------|-------------|
| `totalSales` | number | Total revenue from completed orders (COP) |
| `totalOrders` | number | Total orders (all statuses) |
| `completedOrders` | number | Completed orders count |
| `cancelledOrders` | number | Cancelled orders count |
| `pendingOrders` | number | Pending orders count |
| `avgTicket` | number | Average ticket value for completed orders (COP) |

**`groupBy: "date"` or `"weekday"` or `"hour"`** — each item also includes:

| Field | Type | Description |
|-------|------|-------------|
| `dayCount` | number | Number of distinct days with activity in the period |
| `avgOrdersPerDay` | number | Average orders per active day |
| `avgSalesPerDay` | number | Average revenue per active day |

In [ ]:
# ── Example 1: Overall metrics for the current month ──────────────────────────
first_day_of_month = datetime.now().replace(day=1).strftime("%Y-%m-%d")
today = datetime.now().strftime("%Y-%m-%d")

result = make_request("/sales/metrics", {
    "dateFrom": first_day_of_month,
    "dateTo": today,
    "timezone": "America/Bogota"
})

if result.get("success"):
    data = result['data']
    print("=== MONTHLY METRICS ===")
    print(f"Total Sales:       ${data.get('totalSales', 0):,.0f} COP")
    print(f"Completed Orders:  {data.get('completedOrders', 0)}")
    print(f"Cancelled Orders:  {data.get('cancelledOrders', 0)}")
    print(f"Pending Orders:    {data.get('pendingOrders', 0)}")
    print(f"Avg Ticket:        ${data.get('avgTicket', 0):,.0f} COP")
else:
    print(f"Error: {result.get('error', {}).get('message')}")

# ── Example 2: Day-by-day breakdown ───────────────────────────────────────────
result = make_request("/sales/metrics", {
    "dateFrom": first_day_of_month,
    "dateTo": today,
    "groupBy": "date"
})

if result.get("success"):
    print("\n=== DAILY BREAKDOWN ===")
    for day in result['data']:
        print(f"  {day['date']} ({day['dayName']}): {day['totalOrders']} orders | ${day['totalSales']:,.0f}")

# ── Example 3: By day of week ─────────────────────────────────────────────────
result = make_request("/sales/metrics", {
    "dateFrom": first_day_of_month,
    "dateTo": today,
    "groupBy": "weekday"
})

if result.get("success"):
    print("\n=== BY WEEKDAY ===")
    for d in result['data']:
        print(f"  {d['dayName']}: {d['totalOrders']} orders | avg/day: {d['avgOrdersPerDay']} | ${d['avgSalesPerDay']:,.0f}/day")

# ── Example 4: By hour of day ─────────────────────────────────────────────────
result = make_request("/sales/metrics", {
    "dateFrom": first_day_of_month,
    "dateTo": today,
    "groupBy": "hour"
})

if result.get("success"):
    print("\n=== BY HOUR ===")
    for h in result['data']:
        print(f"  {h['hourLabel']}: {h['totalOrders']} orders | ${h['totalSales']:,.0f}")

# ── Example 5: Top 10 products by revenue ─────────────────────────────────────
result = make_request("/sales/metrics", {
    "dateFrom": first_day_of_month,
    "dateTo": today,
    "groupBy": "product",
    "limit": 10,
    "sortBy": "revenue"
})

if result.get("success"):
    print("\n=== TOP PRODUCTS ===")
    for p in result['data']:
        print(f"  #{p['rank']} {p['productName']} ({p['categoryName']}): {p['totalQuantity']:.0f} units | ${p['totalRevenue']:,.0f}")

# ── Example 6: By payment method ──────────────────────────────────────────────
result = make_request("/sales/metrics", {
    "dateFrom": first_day_of_month,
    "dateTo": today,
    "groupBy": "payment"
})

if result.get("success"):
    print("\n=== BY PAYMENT METHOD ===")
    for m in result['data']:
        print(f"  {m['paymentMethod']}: {m['ordersCount']} orders ({m['ordersPercentage']}%) | ${m['totalSales']:,.0f} ({m['salesPercentage']}%)")

# ── Example 7: Ticket distribution ────────────────────────────────────────────
result = make_request("/sales/metrics", {
    "dateFrom": first_day_of_month,
    "dateTo": today,
    "groupBy": "ticket",
    "ranges": [0, 15000, 25000, 40000, 60000, 100000]
})

if result.get("success"):
    print("\n=== TICKET DISTRIBUTION ===")
    for r in result['data']:
        print(f"  {r['range']}: {r['ordersCount']} orders ({r['percentage']}%) | avg ${r['avgTicket']:,.0f}")

---
## 5. Get Sale Detail

**Endpoint:** `POST /v1/sales/detail`

Retrieves detailed information about a specific sale including all items and modifiers.

**Required Scope:** `orders:read` or `read`

### Parameters

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `orderId` | string | **Yes** | The UUID of the order to retrieve |

### Response Fields

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Order UUID |
| `orderNumber` | number | Human-readable order number |
| `orderDate` | string | Order timestamp (ISO 8601) |
| `totalAmount` | number | Total order value (COP) |
| `status` | string | `"completed"`, `"cancelled"`, or `"pending"` |
| `paymentMethod` | string | `"cash"`, `"card"`, or `"digital"` |
| `customer` | object | `{ id, name, phone }` |
| `items` | array | Array of order items (see below) |

### Item Object Fields

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Order item UUID |
| `quantity` | number | Quantity ordered |
| `priceAtPurchase` | number | Unit price at time of purchase (COP) |
| `subtotal` | number | Line total (COP) |
| `product` | object | `{ id, name }` — product at time of purchase |
| `modifiers` | array | Selected modifiers: `[{ name, price, quantity }]` |

In [ ]:
# First get an order ID from the sales list
sales = make_request("/sales", {"limit": 1})

if sales.get("success") and sales['data']:
    order_id = sales['data'][0]['id']
    print(f"Querying order detail: {order_id}\n")

    detail = make_request("/sales/detail", {"orderId": order_id})

    if detail.get("success"):
        data = detail['data']
        print(f"=== ORDER #{data.get('orderNumber')} ===")
        print(f"Date:           {data.get('orderDate')}")
        print(f"Status:         {data.get('status')}")
        print(f"Payment:        {data.get('paymentMethod')}")
        print(f"Total:          ${data.get('totalAmount', 0):,.0f} COP")
        print(f"Customer:       {data.get('customer', {}).get('name', 'N/A')}")
        print(f"\nItems:")

        for item in data.get('items', []):
            product_name = item['product']['name']  # nested under 'product'
            print(f"  - {product_name} x{item.get('quantity', 1)} | ${item.get('subtotal', 0):,.0f} COP")
            for mod in item.get('modifiers', []):
                print(f"      + {mod.get('name', 'N/A')} (${mod.get('price', 0):,.0f})")
    else:
        print(f"Error: {detail.get('error', {}).get('message')}")
else:
    print("No sales available to query")

---
## 6. Get Menu Products

**Endpoint:** `POST /v1/menu/products`

Retrieves the list of menu products with their ingredients, recipe bases, and modifier groups.

**Required Scope:** `menu:read` or `read`

### Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `limit` | int | 50 | Number of results to return (1-250) |
| `offset` | int | 0 | Number of results to skip for pagination |
| `categoryId` | string | null | Filter by category UUID |
| `isAvailable` | boolean | null | Filter by availability (`true` or `false`) |
| `includeIngredients` | boolean | true | Include direct product ingredients in response |
| `includeRecipeBases` | boolean | true | Include associated recipe bases in response |
| `includeModifiers` | boolean | true | Include associated modifier groups in response |

### Response Fields (per product)

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Product UUID |
| `name` | string | Product name |
| `description` | string | Product description (nullable) |
| `price` | number | Product price in COP |
| `isAvailable` | boolean | Whether product is currently available |
| `allowModifiers` | boolean | Whether product accepts modifier selections |
| `preparationTime` | number | Estimated preparation time in minutes (nullable) |
| `calculatedCost` | number | Calculated ingredient cost in COP (nullable) |
| `category` | object | `{ id, name }` or `null` |
| `ingredients` | array | Direct ingredients (if `includeIngredients: true`) |
| `recipeBases` | array | Recipe bases with their ingredients (if `includeRecipeBases: true`) |
| `modifierGroups` | array | Modifier groups with options (if `includeModifiers: true`) |

### Ingredient Object Fields

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Ingredient UUID |
| `name` | string | Ingredient name |
| `quantity` | number | Amount used |
| `unit` | string | Unit of measure (kg, g, ml, l, unit, etc.) |
| `isRequired` | boolean | Whether ingredient is required for the recipe |

In [ ]:
# Example: Get all available products with full details
result = make_request("/menu/products", {
    "limit": 50,
    "isAvailable": True,
    "includeIngredients": True,
    "includeRecipeBases": True,
    "includeModifiers": True
})

if result.get("success"):
    print(f"Total products: {result['pagination']['total']}\n")

    for product in result['data'][:5]:  # Show only first 5
        print(f"\n=== {product['name']} ===")
        print(f"Price:          ${product.get('price', 0):,.0f} COP")
        print(f"Category:       {product.get('category', {}).get('name', 'N/A') if product.get('category') else 'N/A'}")
        print(f"Available:      {product.get('isAvailable')}")
        cost = product.get('calculatedCost')
        print(f"Calculated Cost: ${cost:,.0f} COP" if cost is not None else "Calculated Cost: N/A")

        # Direct ingredients — field is 'name', not 'ingredientName'
        if product.get('ingredients'):
            print("Ingredients:")
            for ing in product['ingredients'][:3]:
                print(f"  - {ing['name']}: {ing['quantity']} {ing['unit']}")

        if product.get('modifierGroups'):
            print(f"Modifier groups: {len(product['modifierGroups'])}")
else:
    print(f"Error: {result.get('error', {}).get('message')}")

---
## 7. Get Recipe Bases

**Endpoint:** `POST /v1/menu/recipes`

Retrieves the list of recipe bases (base preparations) with their ingredients.

**Required Scope:** `menu:read` or `read`

### Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `limit` | int | 50 | Number of results to return (1-250) |
| `offset` | int | 0 | Number of results to skip for pagination |
| `isActive` | boolean | null | Filter by active status (`true` or `false`) |

### Response Fields (per recipe)

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Recipe UUID |
| `name` | string | Recipe name |
| `description` | string | Recipe description (nullable) |
| `isActive` | boolean | Whether recipe is active |
| `createdAt` | string | Creation timestamp (ISO 8601) |
| `updatedAt` | string | Last update timestamp (ISO 8601) |
| `ingredients` | array | Array of ingredients with quantities |

### Ingredient Object Fields

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Ingredient UUID |
| `name` | string | Ingredient name |
| `quantity` | number | Amount used |
| `unit` | string | Unit of measure (kg, g, ml, l, unit, etc.) |
| `isRequired` | boolean | Whether ingredient is required |
| `notes` | string | Preparation notes for this ingredient (nullable) |

In [ ]:
# Example: Get all active recipes
result = make_request("/menu/recipes", {
    "limit": 50,
    "isActive": True
})

if result.get("success"):
    print(f"Total recipes: {result['pagination']['total']}\n")

    for recipe in result['data']:
        print(f"\n=== {recipe['name']} ===")
        print(f"Status:    {'Active' if recipe.get('isActive') else 'Inactive'}")
        if recipe.get('description'):
            print(f"Description: {recipe['description']}")
        print(f"Updated:   {recipe.get('updatedAt', 'N/A')}")

        if recipe.get('ingredients'):
            print("Ingredients:")
            for ing in recipe['ingredients']:
                # Field is 'name', not 'ingredientName'
                required = " (required)" if ing.get('isRequired') else ""
                notes = f" — {ing['notes']}" if ing.get('notes') else ""
                print(f"  - {ing['name']}: {ing['quantity']} {ing['unit']}{required}{notes}")
else:
    print(f"Error: {result.get('error', {}).get('message')}")

---
## 8. Get Modifier Groups

**Endpoint:** `POST /v1/menu/modifiers`

Retrieves the list of modifier groups with their options and ingredients.

**Required Scope:** `menu:read` or `read`

### Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `limit` | int | 50 | Number of results to return (1-250) |
| `offset` | int | 0 | Number of results to skip for pagination |

### Response Fields (per group)

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Modifier group UUID |
| `name` | string | Group name (e.g., "Extra Toppings") |
| `minQty` | int | Minimum options customer must select |
| `maxQty` | int | Maximum options customer can select |
| `isRequired` | boolean | Whether a selection is mandatory |
| `createdAt` | string | Creation timestamp (ISO 8601) |
| `updatedAt` | string | Last update timestamp (ISO 8601) |
| `modifiers` | array | Array of modifier options (see below) |
| `associatedProducts` | array | Products using this group: `[{ id, name }]` |

### Modifier Option Fields

| Field | Type | Description |
|-------|------|-------------|
| `id` | string | Modifier UUID |
| `name` | string | Option name |
| `price` | number | Price added to order when selected (COP) |
| `isAvailable` | boolean | Whether option is currently available |
| `isDefault` | boolean | Whether option is pre-selected by default |
| `sortOrder` | number | Display order within the group |
| `ingredient` | object | Single ingredient object `{ id, name, quantity, unit }` or `null` |

In [ ]:
# Example: Get all modifier groups
result = make_request("/menu/modifiers", {
    "limit": 50
})

if result.get("success"):
    print(f"Total modifier groups: {result['pagination']['total']}\n")

    for group in result['data']:
        print(f"\n=== {group['name']} ===")
        # Fields are minQty / maxQty (not minSelections / maxSelections)
        print(f"Required: {group.get('isRequired')} | Select: {group.get('minQty', 0)}–{group.get('maxQty', '∞')}")

        print("Options:")
        for mod in group.get('modifiers', []):
            # Field is 'price' (not 'priceAdjustment')
            price = mod.get('price', 0)
            price_str = f"+${price:,.0f}" if price > 0 else "Included"
            default_str = " [default]" if mod.get('isDefault') else ""
            # 'ingredient' is a single object (not an array)
            ing = mod.get('ingredient')
            ing_str = f" — uses {ing['name']}" if ing else ""
            print(f"  - {mod['name']} ({price_str}){default_str}{ing_str}")

        # Products that use this modifier group
        associated = group.get('associatedProducts', [])
        if associated:
            print(f"Used by: {', '.join(p['name'] for p in associated)}")
else:
    print(f"Error: {result.get('error', {}).get('message')}")

---
## 9. Pagination - Get All Results

When there are more results than the maximum limit (250), you need to paginate through the results.

### Pagination Response Fields

| Field | Type | Description |
|-------|------|-------------|
| `total` | int | Total number of results available |
| `limit` | int | Number of results per page |
| `offset` | int | Current offset position |
| `hasMore` | boolean | Whether there are more results to fetch |

In [ ]:
def get_all_results(endpoint, params=None):
    """
    Function to get all results with automatic pagination.
    
    Args:
        endpoint: The endpoint to query
        params: Additional filter parameters
    
    Returns:
        list: List with all results
    """
    all_results = []
    offset = 0
    limit = 250  # Maximum allowed
    
    while True:
        data = {"limit": limit, "offset": offset}
        if params:
            data.update(params)
        
        result = make_request(endpoint, data)
        
        if not result.get("success"):
            print(f"Error: {result.get('error', {}).get('message')}")
            break
        
        all_results.extend(result['data'])
        
        # Check if there are more results
        if not result['pagination'].get('hasMore', False):
            break
        
        offset += limit
        print(f"Retrieved {len(all_results)} of {result['pagination']['total']}...")
    
    return all_results

# Example: Get all sales from the month
# all_sales = get_all_results("/sales", {
#     "dateFrom": "2025-01-01",
#     "dateTo": "2025-01-31"
# })
# print(f"Total sales retrieved: {len(all_sales)}")

print("Pagination function ready. Uncomment the code to use it.")

---
## Important Notes

### Rate Limiting

Requests may be subject to rate limiting at the infrastructure level. Recommended limits to stay within:
- ~100 requests per minute
- ~1,000 requests per hour

Rate limit headers may be included in responses:
- `X-RateLimit-Limit`
- `X-RateLimit-Remaining`
- `X-RateLimit-Reset`

### Token Scopes

| Scope | Access |
|-------|--------|
| `orders:read` | Sales endpoints (`/sales`, `/sales/metrics`, `/sales/detail`) |
| `menu:read` | Menu endpoints (`/menu/products`, `/menu/recipes`, `/menu/modifiers`) |
| `read` | Full read access to all endpoints |
| `write` | Full read + write access |

### Meta Object

Every API response includes a `meta` field with request context:

```json
{
  "meta": {
    "tokenId": "uuid",
    "tenantId": "uuid",
    "timezone": "America/Bogota"
  }
}
```

### Supported Timezones

| Timezone | Region |
|----------|--------|
| `America/Bogota` | Colombia (default) |
| `America/Mexico_City` | Mexico |
| `America/New_York` | US Eastern |
| `America/Los_Angeles` | US Pacific |
| `Europe/Madrid` | Spain |
| `Europe/London` | UK |

### groupBy Quick Reference (`/sales/metrics`)

| `groupBy` | Key response fields |
|-----------|---------------------|
| `null` | `totalSales`, `totalOrders`, `completedOrders`, `cancelledOrders`, `pendingOrders`, `avgTicket` |
| `"date"` | `date`, `dayName`, `totalOrders`, `totalSales`, `avgTicket` |
| `"weekday"` | `dayNumber`, `dayName`, `dayCount`, `totalOrders`, `totalSales`, `avgTicket`, `avgOrdersPerDay`, `avgSalesPerDay` |
| `"hour"` | `hour`, `hourLabel`, `dayCount`, `totalOrders`, `totalSales`, `avgTicket`, `avgOrdersPerDay`, `avgSalesPerDay` |
| `"product"` | `rank`, `productId`, `productName`, `categoryName`, `totalQuantity`, `totalRevenue`, `ordersCount`, `avgPrice` |
| `"payment"` | `paymentMethod`, `ordersCount`, `ordersPercentage`, `totalSales`, `salesPercentage`, `avgTicket` |
| `"ticket"` | `range`, `ordersCount`, `percentage`, `totalSales`, `avgTicket` |